# Namelyze Tutorial

**Scholar Nationality and Gender Inference Tool - Complete Workflow Guide**

This tutorial will guide you step-by-step on how to use Namelyze to infer the nationality and gender of scholar names.

**Contact Information:**
- **Name:** Liyuan Shang
- **Institution:** Fudan University  
- **Email:** lyshang23@m.fudan.edu.cn

## 📋 Table of Contents

1.  [Environment Setup](#1-environment-setup)
2.  [API Configuration](#2-api-configuration)
3.  [Data Preparation](#3-data-preparation)
4.  [Running Inference](#4-running-inference)
5.  [Result Analysis](#5-result-analysis)
6.  [Advanced Usage](#6-advanced-usage)
7.  [Frequently Asked Questions](#7-frequently-asked-questions)

## 1. Environment Setup

### 1.1 Checking Python Version

In [ ]:
import sys
print(f"Python version: {sys.version}")

# Ensure Python version >= 3.8
assert sys.version_info >= (3, 8), "Python 3.8 or higher is required"
print("✓ Python version meets requirements")

### 1.2 Installing Dependencies

In [ ]:
# Install required Python packages
!pip install openai pandas python-dotenv pydantic pydantic-settings tqdm tenacity -q
print("✓ Dependencies installed successfully")

### 1.3 Verifying Installation

In [ ]:
import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import __version__ as pydantic_version
import pycountry
from pathlib import Path

print("✓ All dependencies imported successfully")
print(f"  - openai: {openai.__version__}")
print(f"  - pandas: {pd.__version__}")
print(f"  - pydantic: {pydantic_version}")
print(f"  - pycountry: {pycountry.__version__}")

## 2. API Configuration

### 2.1 Creating Configuration File

First, we need to configure the API information. You can use any OpenAI-compatible API service.

Depending on the API provider you use, obtain the following three parameters. Configuration examples are shown below:

**OpenAI**:
```bash
OPENAI_API_BASE=https://api.openai.com/v1
OPENAI_API_KEY=sk-your-key-here
MODEL_NAME=gpt-5-nano
```

**DeepSeek**:
```bash
OPENAI_API_BASE=https://api.deepseek.com
OPENAI_API_KEY=your-deepseek-key
MODEL_NAME=deepseek-chat
```

In [ ]:
OPENAI_API_BASE = "https://api.deepseek.com"
OPENAI_API_KEY = "sk-59ce710eecd844a8bdbff0957b5ac415"
MODEL_NAME =     "deepseek-chat"
BATCH_SIZE = 2
MAX_WORKERS= 2
ENABLE_CONCURRENT = True
INPUT_CSV =  "data/input/names_small.csv"
OUTPUT_CSV=  "data/output/results.csv"
NAME_COLUMN= "name"
# .env file
env_content = f"""# OpenAI Compatible API Configuration
OPENAI_API_BASE={OPENAI_API_BASE}
OPENAI_API_KEY={OPENAI_API_KEY}
MODEL_NAME={MODEL_NAME}

# Processing Configuration
BATCH_SIZE={BATCH_SIZE}
MAX_RETRIES=3
TIMEOUT=60
MAX_WORKERS={MAX_WORKERS}
ENABLE_CONCURRENT={ENABLE_CONCURRENT}

# File Paths
INPUT_CSV={INPUT_CSV}
OUTPUT_CSV={OUTPUT_CSV}
NAME_COLUMN={NAME_COLUMN}
"""

In [ ]:
env_file = Path(".env")
# create .env file
with open(env_file, 'w', encoding='utf-8') as f:
    f.write(env_content)

### 2.2 Verifying Configuration

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Check required configuration items
api_key = os.getenv("OPENAI_API_KEY")
api_base = os.getenv("OPENAI_API_BASE")
model_name = os.getenv("MODEL_NAME")

print("Configuration check:")
print(f"  API Base: {api_base}")
print(f"  API Key: {api_key[:10]}..." if api_key else "  API Key: Not set ❌")
print(f"  Model: {model_name}")

if not api_key or api_key == "sk-your-api-key-here":
    print("\n⚠️  Please set a valid API key in the .env file first!")
else:
    print("\n✓ Configuration verified successfully")

## 3. Data Preparation

In [ ]:
import pandas as pd

# Read sample data
sample_df = pd.read_csv(Path(INPUT_CSV))
print(f"Sample data contains {len(sample_df)} scholar names:")
print()
sample_df

## 4. Running Inference

### 4.1 Importing Modules

In [ ]:
import sys
sys.path.append('.')

from src.config import load_settings
from src.llm_client import LLMClient
from src.processor import ScholarProcessor

print("✓ Modules imported successfully")

### 4.2 System Initialization

In [ ]:
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Load settings
settings = load_settings()

# Initialize LLM client
llm_client = LLMClient(
    api_base=settings.openai_api_base,
    api_key=settings.openai_api_key,
    model_name=settings.model_name,
    timeout=settings.timeout,
    max_retries=settings.max_retries
)

# Initialize processor
processor = ScholarProcessor(
    llm_client=llm_client,
    batch_size=settings.batch_size,
    max_workers=settings.max_workers,
    enable_concurrent=settings.enable_concurrent
)

print("✓ System initialization completed")

### 4.3 Running the Inference Process

In [ ]:
# Execute the complete processing pipeline
processor.run(
    input_csv=settings.input_csv,
    output_csv=settings.output_csv,
    name_column=settings.name_column
)

print("\n✓ Processing completed!")

## 5. Result Analysis

### 5.1 Viewing Results

In [ ]:
# Read the result file
results_df = pd.read_csv(settings.output_csv)

print(f"Processing results contain {len(results_df)} records:")
print()
results_df

### 5.2 Statistical Analysis

In [ ]:
# Count successful and failed results
success_count = (results_df['has_error'] == 'No').sum()
error_count = (results_df['has_error'] == 'Yes').sum()

print(f"Processing Statistics:")
print(f"  ✓ Success: {success_count} ({success_count/len(results_df)*100:.1f}%)")
print(f"  ✗ Errors: {error_count} ({error_count/len(results_df)*100:.1f}%)")

# Gender distribution
print(f"\nGender Distribution:")
gender_counts = results_df['gender'].value_counts()
for gender, count in gender_counts.items():
    print(f"  {gender}: {count}")

# Confidence distribution
print(f"\nGender Confidence Distribution:")
conf_gender_counts = results_df['conf_gender'].value_counts()
for conf, count in conf_gender_counts.items():
    print(f"  {conf}: {count}")

print(f"\nNationality Confidence Distribution:")
conf_nation_counts = results_df['conf_nation'].value_counts()
for conf, count in conf_nation_counts.items():
    print(f"  {conf}: {count}")

### 5.3 Viewing Error Records

In [ ]:
# Filter records with errors
error_records = results_df[results_df['has_error'] == 'Yes']

if len(error_records) > 0:
    print(f"Found {len(error_records)} error records:")
    print()
    display(error_records[['name', 'error_reason']])
else:
    print("✓ No error records found")

## 6. Advanced Usage

### 6.1 Viewing Prompt Templates

In [ ]:
from src.prompt_template import generate_prompt

# Generate a sample prompt
sample_prompt = generate_prompt(["Adam Smith", "李四"])

print("Prompt Template Content:")
print("=" * 60)
print(sample_prompt)  # Only showing first 1000 characters
print("...")
print("=" * 60)

## 7. Frequently Asked Questions

### Q1: What should I do if API calls fail?

**A:** Check the following points:
- Is the API key correct?
- Is the API Base URL correct?
- Is the network connection normal?
- Have you triggered API rate limiting?

The system will automatically retry failed requests (default: 3 times).

### Q2: Why are some results marked as errors?

**A:** Possible reasons:
- LLM returned incorrect JSON format
- Some fields are missing
- Field values do not match the expected format

Check the `error_reason` column for specific details.

### Q3: How can I improve accuracy?

**A:** Recommendations:
- Use more powerful models (e.g., GPT-4 instead of GPT-3.5)
- For important data, manually review low-confidence results
- Consider using multiple models for cross-validation

### Q4: How should I choose the batch size?

**A:** Recommendations:
- Default value of 20 is suitable for most situations
- If you encounter token limits, reduce the batch size
- If the model supports larger context, you can increase the batch size to improve efficiency

### Q5: Which country codes are supported?

**A:** Uses ISO 3166-1 alpha-3 standard (three-letter codes), such as:
- USA (United States)
- CHN (China)
- GBR (United Kingdom)
- JPN (Japan)
- etc.

Check the `VALID_NATIONS` set in `src/validator.py` for the complete list.

## Summary

Congratulations! You have completed the full Namelyze tutorial.

### Key Takeaways:

1. ✅ Configure correct API information
2. ✅ Prepare CSV input files in standard format
3. ✅ Run inference and obtain results
4. ✅ Analyze results and pay attention to error records
5. ✅ Adjust batch size and model parameters as needed

### Next Steps:

- Test with your own data
- Adjust configurations according to your actual needs
- Check README.md for more detailed information
- Check the log file `namelyze.log` if you encounter any issues

Wishing you a pleasant experience using the tool!